In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_excel('../data/raw/Online Retail.xlsx')

# First look
print(df.shape)
print(df.head())

(541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [2]:
print(pd.__version__)

3.0.1


In [4]:
# Initial inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nBasic Stats:\n", df.describe())

Shape: (397884, 11)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Month', 'Year']

Data Types:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
Revenue               float64
Month               period[M]
Year                    int32
dtype: object

Missing Values:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
Month          0
Year           0
dtype: int64

Duplicate Rows: 5192

Basic Stats:
             Quantity                 InvoiceDate      UnitPrice  \
count  397884.000000                      397884  397884.000000   
mean       12.988238  2011-07-10 23:41:23.511023       3.116488   
min         1.000000 

In [5]:
# StockCode is a product ID - not needed for sales analysis
# We keep all other columns as they are all useful
print("Columns before:", df.columns.tolist())

# No columns to drop entirely - all are useful
# But let's confirm no constant/useless columns exist
print("\nUnique values per column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

Columns before: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Month', 'Year']

Unique values per column:
InvoiceNo: 18532 unique values
StockCode: 3665 unique values
Description: 3877 unique values
Quantity: 301 unique values
InvoiceDate: 17282 unique values
UnitPrice: 440 unique values
CustomerID: 4338 unique values
Country: 37 unique values
Revenue: 2939 unique values
Month: 13 unique values
Year: 2 unique values


In [6]:
print("Missing before:\n", df.isnull().sum())

# Drop rows where CustomerID is missing (can't do customer analysis without it)
df = df.dropna(subset=['CustomerID'])

# Drop rows where Description is missing
df = df.dropna(subset=['Description'])

print("\nMissing after:\n", df.isnull().sum())
print("Shape after dropping nulls:", df.shape)

Missing before:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
Month          0
Year           0
dtype: int64

Missing after:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
Month          0
Year           0
dtype: int64
Shape after dropping nulls: (397884, 11)


In [7]:
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())
print("Shape after removing duplicates:", df.shape)

Duplicates before: 5192
Duplicates after: 0
Shape after removing duplicates: (392692, 11)


In [8]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Convert CustomerID to integer
df['CustomerID'] = df['CustomerID'].astype(int)

# Convert InvoiceNo to string
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

print("Data types after fixing:\n", df.dtypes)

Data types after fixing:
 InvoiceNo                 str
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
Revenue               float64
Month               period[M]
Year                    int32
dtype: object


In [9]:
print("Shape before:", df.shape)

# Remove cancelled orders (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].str.startswith('C')]
print("After removing cancellations:", df.shape)

# Remove negative or zero quantities
df = df[df['Quantity'] > 0]
print("After removing bad quantities:", df.shape)

# Remove negative or zero unit prices
df = df[df['UnitPrice'] > 0]
print("After removing bad prices:", df.shape)

# Remove test/junk StockCodes (less than 4 characters)
df = df[df['StockCode'].astype(str).str.len() >= 4]
print("After removing junk StockCodes:", df.shape)

# Remove known non-product entries
junk_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df = df[~df['StockCode'].astype(str).isin(junk_codes)]
print("After removing junk entries:", df.shape)

Shape before: (392692, 11)
After removing cancellations: (392692, 11)
After removing bad quantities: (392692, 11)
After removing bad prices: (392692, 11)
After removing junk StockCodes: (392264, 11)
After removing junk entries: (391150, 11)


In [10]:
# Check distribution
print("Quantity stats:\n", df['Quantity'].describe())
print("\nUnitPrice stats:\n", df['UnitPrice'].describe())

# Remove extreme outliers using IQR method for Quantity
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
df = df[df['Quantity'] <= upper_limit]
print(f"\nQuantity upper limit set to: {upper_limit}")

# Remove extreme outliers for UnitPrice
Q1p = df['UnitPrice'].quantile(0.25)
Q3p = df['UnitPrice'].quantile(0.75)
IQRp = Q3p - Q1p
upper_limit_price = Q3p + 3 * IQRp
df = df[df['UnitPrice'] <= upper_limit_price]
print(f"UnitPrice upper limit set to: {upper_limit_price}")

print("\nShape after outlier removal:", df.shape)

Quantity stats:
 count    391150.000000
mean         13.145300
std         180.807831
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

UnitPrice stats:
 count    391150.000000
mean          2.874372
std           4.284738
min           0.040000
25%           1.250000
50%           1.950000
75%           3.750000
max         649.500000
Name: UnitPrice, dtype: float64

Quantity upper limit set to: 42.0
UnitPrice upper limit set to: 11.25

Shape after outlier removal: (364431, 11)


In [11]:
# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Extract date parts
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Month_Year'] = df['InvoiceDate'].dt.to_period('M')
df['Day'] = df['InvoiceDate'].dt.day
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

print("New columns added:", df.columns.tolist())
print("\nFinal shape:", df.shape)
print(df.head())

New columns added: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Month', 'Year', 'Month_Year', 'Day', 'DayOfWeek', 'Hour']

Final shape: (364431, 15)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  Revenue  Month  \
0 2010-12-01 08:26:00       2.55       17850  United Kingdom    15.30     12   
1 2010-12-01 08:26:00       3.39       17850  United Kingdom    20.34     12   
2 2010-12-01 08:26:00       2.75       17850  United Kingdom    22.00     12   
3 2010-12-01 08:26:00   

In [12]:
# Save cleaned dataset for Power BI
df.to_csv('../data/cleaned/cleaned_retail.csv', index=False)
print("✅ Cleaned data saved to data/cleaned/cleaned_retail.csv")
print(f"Final dataset: {df.shape[0]:,} rows and {df.shape[1]} columns")

✅ Cleaned data saved to data/cleaned/cleaned_retail.csv
Final dataset: 364,431 rows and 15 columns


In [13]:
import numpy as np

# Reference date (day after last transaction)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate RFM metrics per customer
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()

# Score each metric 1-4
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4])

# Combined RFM score
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Segment customers
def segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalist'
    elif r == 2:
        return 'At Risk'
    elif r == 1 and f >= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment, axis=1)

print(rfm.head(10))
print("\nSegment Distribution:")
print(rfm['Segment'].value_counts())

# Save RFM table
rfm.to_csv('../data/cleaned/rfm_segments.csv', index=False)
print("\n✅ RFM data saved!")

   CustomerID  Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Score  \
0       12347        2          7   3783.23       4       4       4       444   
1       12348      249          3     90.20       1       3       1       131   
2       12349       19          1   1328.55       3       1       3       313   
3       12350      310          1    294.40       1       1       2       112   
4       12352       36          7   1321.99       3       4       3       343   
5       12353      204          1     89.00       1       1       1       111   
6       12354      232          1   1013.60       1       1       3       113   
7       12355      214          1    238.90       1       1       1       111   
8       12356       23          3   1778.89       3       3       4       334   
9       12357       33          1   4817.23       3       1       4       314   

              Segment  
0            Champion  
1                Lost  
2  Potential Loyalist  
3     Needs 

In [15]:
print(rfm['Segment'].value_counts().to_string())

Segment
At Risk               1040
Loyal                 1004
Potential Loyalist     643
Lost                   549
Needs Attention        504
Champion               488
